# Task Retries, Stage Retries, and Failure Handling

**Module**: 11_fault_tolerance_and_reliability  
**Notebook**: 03_task_retries.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Task retry mechanisms, spark.task.maxFailures, stage retries, and handling transient exceptions.
- **Why does this topic exist?**: Transient errors (network hiccups, disk blips) should not abort long-running ETL jobs.
- **What problem does it solve?**: Understanding task retry limits and avoiding poison pill data crashes.
- **Where is it used in real-world Data Engineering?**: Production resilience engineering.

---

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Task_Retries_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
print("spark.task.maxFailures: ", spark.conf.get("spark.task.maxFailures", "4 (default)"))

---

## Experiment: Simulating a Task Retry with an Unstable UDF

- **1. What are we testing?**: Run a function that fails on attempt 0 and succeeds on attempt 1.
- **2. What do we expect?**: Spark retries the task automatically up to maxFailures.

In [ ]:
from pyspark import TaskContext

def flaky_task(x):
    ctx = TaskContext.get()
    attempt = ctx.attemptNumber()
    if x == 42 and attempt == 0:
        raise RuntimeError("Transient blip on attempt 0!")
    return f"Processed {x} on attempt {attempt}"

try:
    res = spark.sparkContext.parallelize([1, 42, 3], 2).map(flaky_task).collect()
    print("Flaky task succeeded after retry:")
    for r in res:
        print(" ", r)
except Exception as e:
    print(f"Task retry failed: {e}")

**3. What happened?**: Task failed on attempt 0 and automatically succeeded on attempt 1.  
**4. Why did it happen?**: TaskScheduler re-queued the task because attemptNumber < spark.task.maxFailures.  
**5. Practical Engineering Lesson**: Spark task retries absorb transient network and I/O glitches seamlessly.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Max Failures
**Problem Statement**: Print spark.task.maxFailures.  
**Input / Context**: conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Poison Pill Concept
**Problem Statement**: Explain what a poison pill record is.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Stage Abort Rule
**Problem Statement**: When is an entire stage aborted?  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Max Failures):
```python
print(spark.conf.get('spark.task.maxFailures','4'))
```

#### Solution 2 (Poison Pill Concept):
```python
print('A bad record that deterministically crashes on every retry until maxFailures is exceeded.')
```

#### Solution 3 (Stage Abort Rule):
```python
print('When any single task fails 4 consecutive times.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Task Retries**: Default 4 attempts per task.
- **Stage Resilience**: Stage retries if shuffle fetch fails.

### Essential Technical Terms
- `attemptNumber`: Current retry attempt index.
- `Poison Pill`: Deterministic failure record.

### Common Engineering Mistakes
- **Setting maxFailures=1 in production**: Any minor network blip aborts the entire job.

### Interview Questions on This Topic
- **Q: How many times will Spark retry a failed task by default?**
  - *A*: 4 times (spark.task.maxFailures = 4).

### What to Study Next
Open 04_checkpointing.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')